# META-CXR Training on Kaggle (2x T4 GPU)

This notebook trains the META-CXR model on the MIMIC-CXR-JPG dataset using 2x T4 GPUs via PyTorch DistributedDataParallel.

**Prerequisites:**
- Kaggle accelerator set to **GPU T4 x2**
- Two datasets attached as Kaggle input:
  - **`mimic-cxr-jpg-lite`** — JPG images + metadata CSVs (images at `p10/...`)
  - **`mimic-cxr-reported`** — radiology `.txt` reports (at `files/p10.../`)
- Internet access enabled

**Steps:** Run cells 1→6 in order.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Kaggle has PyTorch, torchvision, numpy, pandas, scikit-learn preinstalled.
# Install only the packages that are missing.
packages = [
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "scikit-image",           # latest stable; io/transform APIs are unchanged
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal",       # provides health_multimodal used by biovil_t
    "timm==0.6.13",           # 0.6.x keeps timm.models.hub API used by dist_utils.py; PyTorch 2.x compatible
    "spacy",                  # latest 3.x; stable spacy.load() API
    "nltk==3.8.1",
    "google-cloud-storage",
    "transformers==4.44.2",   # pin for Qformer.py compatibility (apply_chunking_to_forward et al.)
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

# Install peft at the specific commit used by the project
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08"],
    check=True
)

# Download NLTK data required by the METEOR scorer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Download spacy English model required by blip2.py (spacy.load("en_core_web_sm"))
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

# Detect Java installation (needed for METEOR/ROUGE scoring)
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
JAVA_HOME_DETECTED = result.stdout.strip()
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

# Verify GPU count
import torch
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## Cell 2 — Weights & Biases Setup

Đăng nhập wandb bằng Kaggle Secret `WANDB_API_KEY`.

**Cách thêm secret trên Kaggle:**  
Notebook Settings → Add-ons → Secrets → **Name:** `WANDB_API_KEY` → **Value:** API key của bạn.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("wandb: API key loaded from Kaggle Secrets")
except Exception as e:
    print(f"wandb: Could not load from Kaggle Secrets ({e}) — using pre-configured key if available")

import wandb
wandb.login()
print("wandb: Logged in successfully")

## Cell 3 — Clone GitHub Repository

In [ ]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}, pulling latest changes...")
    !git -C {REPO_DIR} pull

# Change working directory to repo root
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

## Cell 4 — Verify Kaggle Input Datasets & Load Reports CSV

Two datasets must be attached to this notebook:

| Dataset | Slug | Contents |
|---------|------|----------|
| MIMIC-CXR-JPG-LITE | `mimic-cxr-jpg-lite` | JPG images at `p10/…` + metadata CSVs |
| mimic-cxr-reported | `mimic-cxr-reported` | `mimic_cxr_cleaned.csv` (pre-built) + `.txt` reports |

`mimic_cxr_cleaned.csv` đã được tạo sẵn trong dataset `mimic-cxr-reported`. Cell này chỉ verify paths và load file CSV, không cần rebuild mỗi session.

In [ ]:
import os
import glob
import shutil
import yaml

# ── Load Kaggle dataset config ────────────────────────────────────────────────
with open("configs/kaggle_datasets.yaml") as f:
    CFG = yaml.safe_load(f)

IMAGES_SLUG   = CFG["datasets"]["images"]["slug"]
REPORTS_SLUG  = CFG["datasets"]["reports"]["slug"]
REQUIRED_CSVS = CFG["datasets"]["images"]["required_files"]
CLEANED_CSV   = CFG["datasets"]["reports"]["cleaned_csv_filename"]
REPORTS_LOCAL = CFG["working"]["cleaned_csv_path"]

def find_mount(slug):
    for root in CFG["mount_search_roots"]:
        candidate = os.path.join(root, slug)
        if os.path.isdir(candidate):
            return candidate
    matches = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    return matches[0] if matches else None

# ── Images + metadata CSVs ───────────────────────────────────────────────────
KAGGLE_INPUT = find_mount(IMAGES_SLUG)
if not KAGGLE_INPUT:
    raise FileNotFoundError(
        f"Dataset '{IMAGES_SLUG}' not attached. Add it via Kaggle: Add Data → Datasets."
    )
os.environ["KAGGLE_INPUT"] = KAGGLE_INPUT
os.environ["IMAGE_ROOT"]   = KAGGLE_INPUT
print(f"KAGGLE_INPUT (images + CSVs): {KAGGLE_INPUT}")

# ── Reports dataset ───────────────────────────────────────────────────────────
REPORTS_ROOT = find_mount(REPORTS_SLUG)
if not REPORTS_ROOT:
    raise FileNotFoundError(
        f"Dataset '{REPORTS_SLUG}' not attached. Add it via Kaggle: Add Data → Datasets."
    )
os.environ["REPORTS_ROOT"] = REPORTS_ROOT
print(f"REPORTS_ROOT:                 {REPORTS_ROOT}")

# ── Verify metadata CSVs ─────────────────────────────────────────────────────
for fname in REQUIRED_CSVS:
    path = os.path.join(KAGGLE_INPUT, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {path}")
print("All metadata CSVs present.")

# ── Load pre-built CSV from dataset ──────────────────────────────────────────
CSV_IN_DATASET = os.path.join(REPORTS_ROOT, CLEANED_CSV)

if not os.path.exists(CSV_IN_DATASET):
    raise FileNotFoundError(
        f"{CLEANED_CSV} not found at {CSV_IN_DATASET}.\n"
        "Run generate_mimic_cxr_cleaned.ipynb once to build it, "
        "then upload to the '{REPORTS_SLUG}' Kaggle dataset."
    )

shutil.copy(CSV_IN_DATASET, REPORTS_LOCAL)
os.environ["REPORTS_CSV"] = REPORTS_LOCAL

import pandas as pd
df = pd.read_csv(REPORTS_LOCAL)
print(f"Loaded {REPORTS_LOCAL}: {len(df)} rows")
print(f"\nSample:")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())

## Cell 5 — Write `configs/env_config.yaml` with Kaggle Paths

In [ ]:
import os
import subprocess

result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

KAGGLE_INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input/mimic-cxr-jpg-lite")
IMAGE_ROOT   = os.environ.get("IMAGE_ROOT",   KAGGLE_INPUT)
REPORTS_CSV  = os.environ.get("REPORTS_CSV",  "/kaggle/working/mimic_cxr_cleaned.csv")

env_config_content = f"""paths:
  data_root: \"{KAGGLE_INPUT}\"
  mimic_cxr_jpg_root: \"{IMAGE_ROOT}\"
  split_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-split.csv\"
  reports_csv: \"{REPORTS_CSV}\"
  chexpert_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-chexpert.csv\"
  metadata_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-metadata.csv\"
  output_dir: \"/kaggle/working/output\"
  checkpoint_dir: \"/kaggle/working/checkpoints\"

wandb:
  entity: \"\"
  project: \"meta-cxr\"

java:
  home: \"{java_home}\"
  path: \"{java_path}\"
"""

os.makedirs("configs", exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    f.write(env_config_content)

print("Written configs/env_config.yaml:")
print(env_config_content)


## Cell 6 — Launch 2-GPU DDP Training

Uses `torch.distributed.run` (alias for `torchrun`) with `--standalone` for single-node multi-GPU.  
Training output is streamed live. Expect each epoch to take 30–90 minutes depending on dataset size.

In [ ]:
import subprocess
import sys
import os
import glob
import yaml

os.makedirs("/kaggle/working/output", exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

# ── Resume detection: if checkpoints dataset is attached, pick latest epoch ──
with open("configs/kaggle_datasets.yaml") as f:
    CFG = yaml.safe_load(f)

CKPT_SLUG = CFG["datasets"]["checkpoints"]["slug"]

def find_mount(slug):
    for root in CFG["mount_search_roots"]:
        candidate = os.path.join(root, slug)
        if os.path.isdir(candidate):
            return candidate
    matches = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    return matches[0] if matches else None

resume_args = []
ckpt_root = find_mount(CKPT_SLUG)
if ckpt_root:
    numbered = sorted(
        glob.glob(f"{ckpt_root}/**/checkpoint_[0-9]*.pth", recursive=True),
        key=lambda p: int(os.path.basename(p).rsplit("_", 1)[1].split(".")[0]),
    )
    if numbered:
        resume_path = numbered[-1]
        print(f"Resume from: {resume_path}")
        resume_args = ["--options", f"run.resume_ckpt_path={resume_path}"]
    else:
        print(f"Checkpoint dataset attached at {ckpt_root} but no checkpoint_<N>.pth found — training from scratch.")
else:
    print(f"No '{CKPT_SLUG}' dataset attached — training from scratch (session 1).")

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    "--master_port=12355",
    "-m", "pretraining.train",
    "--cfg-path", "pretraining/configs/mimic_cxr_2gpu.yaml",
] + resume_args

print("Launch command:")
print(" ".join(cmd))
print("\n" + "="*60 + "\n")

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working/META-CXR"

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd="/kaggle/working/META-CXR",
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
print(f"\n" + "="*60)
print(f"Training finished with exit code: {process.returncode}")

## Cell 7 — Display Evaluation Results

In [ ]:
import json
import os
import glob
import pandas as pd

OUTPUT_DIR = "/kaggle/working/output"

# ── Training logs ────────────────────────────────────────────────────────────
log_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/log.txt", recursive=True))
print(f"Found {len(log_files)} log file(s)")

for log_file in log_files:
    print(f"\n{'='*60}")
    print(f"Log: {log_file}")
    print('='*60)
    records = []
    with open(log_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(line)
    if records:
        df = pd.DataFrame(records)
        display(df)

# ── Prediction files ─────────────────────────────────────────────────────────
pred_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/predictions_*.txt", recursive=True))
print(f"\nFound {len(pred_files)} prediction file(s)")

for pred_file in pred_files[:2]:
    print(f"\n{'='*60}")
    print(f"Predictions: {pred_file}")
    print('='*60)
    with open(pred_file) as f:
        for i, line in enumerate(f):
            if i >= 10:
                print(f"  ... ({sum(1 for _ in open(pred_file))} total lines)")
                break
            print(line, end="")

# ── Checkpoint summary ───────────────────────────────────────────────────────
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/**/checkpoint_*.pth", recursive=True))
print(f"\nSaved checkpoints ({len(checkpoints)}):")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / (1024 ** 2)
    print(f"  {ckpt}  ({size_mb:.1f} MB)")

# ── Best checkpoint info ─────────────────────────────────────────────────────
best_ckpts = glob.glob(f"{OUTPUT_DIR}/**/checkpoint_best.pth", recursive=True)
if best_ckpts:
    print(f"\nBest checkpoint: {best_ckpts[0]}")

## Cell 8 — Auto-Push Checkpoints to Kaggle Dataset

Cell này tự động đẩy thư mục `output/` (chứa checkpoints `.pth`) lên một Kaggle Dataset riêng để dùng cho session sau (resume training) hoặc tải về máy.

**Cách hoạt động:**
- Lần đầu (session 1): tạo mới dataset với slug từ `configs/kaggle_datasets.yaml` (`datasets.checkpoints.slug`), mặc định private.
- Lần sau: push version mới lên cùng dataset đó (tự ghi đè bằng version mới nhất).

**Setup một lần:** Kaggle API token đã có sẵn trong kernel của bạn (xem Kaggle → Account → Create New API Token). Không cần thao tác gì thêm.

**Resume ở session sau:**  
Notebook Settings → Add Data → tìm dataset `meta-cxr-checkpoints` của bạn → Add. Cell 6 sẽ tự detect và resume từ checkpoint mới nhất.

**Tải về máy local (optional):**
```bash
kaggle datasets download <username>/meta-cxr-checkpoints -p ./checkpoints --unzip
```

In [ ]:
import os, json, subprocess, glob, yaml

OUTPUT_DIR = "/kaggle/working/output"

with open("configs/kaggle_datasets.yaml") as f:
    CFG = yaml.safe_load(f)

CKPT_CFG = CFG["datasets"]["checkpoints"]

# ── Setup Kaggle API credentials for CLI ──────────────────────────────────────
# Kaggle CLI requires ~/.kaggle/kaggle.json or KAGGLE_USERNAME + KAGGLE_KEY env vars.
# In Kaggle kernels, credentials are NOT auto-placed — load from Kaggle Secrets.

def _setup_kaggle_credentials():
    # Already configured?
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return os.environ["KAGGLE_USERNAME"]

    # Try ~/.kaggle/kaggle.json (exists if user placed it manually)
    kjson = os.path.expanduser("~/.kaggle/kaggle.json")
    if os.path.exists(kjson):
        with open(kjson) as f:
            creds = json.load(f)
        os.environ["KAGGLE_USERNAME"] = creds["username"]
        os.environ["KAGGLE_KEY"] = creds["key"]
        return creds["username"]

    # Load from Kaggle Secrets (recommended)
    try:
        from kaggle_secrets import UserSecretsClient
        sec = UserSecretsClient()
        username = sec.get_secret("KAGGLE_USERNAME")
        key = sec.get_secret("KAGGLE_KEY")
        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key
        # Write kaggle.json so CLI can find it
        os.makedirs(os.path.dirname(kjson), exist_ok=True)
        with open(kjson, "w") as f:
            json.dump({"username": username, "key": key}, f)
        os.chmod(kjson, 0o600)
        return username
    except Exception as e:
        raise RuntimeError(
            "Kaggle API credentials not found.\n"
            "Add KAGGLE_USERNAME and KAGGLE_KEY via Kaggle Secrets:\n"
            "  Notebook Settings → Add-ons → Secrets → '+ Add New Secret'\n"
            f"  (original error: {e})"
        )

USERNAME = _setup_kaggle_credentials()
DATASET_ID = f"{USERNAME}/{CKPT_CFG['slug']}"
print(f"Kaggle username: {USERNAME}")
print(f"Target dataset:  https://www.kaggle.com/datasets/{DATASET_ID}")

# ── List checkpoints to push ─────────────────────────────────────────────────
if not os.path.exists(OUTPUT_DIR):
    print(f"Output dir {OUTPUT_DIR} not found — nothing to push.")
else:
    ckpts = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith(".pth"))
    if not ckpts:
        print(f"No .pth files in {OUTPUT_DIR} — nothing to push.")
    else:
        print(f"\nPushing {len(ckpts)} checkpoint(s):")
        for c in ckpts:
            size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, c)) / (1024 ** 2)
            print(f"  {c}  ({size_mb:.1f} MB)")

        # dataset-metadata.json (required by kaggle CLI)
        metadata = {
            "id": DATASET_ID,
            "title": CKPT_CFG["title"],
            "licenses": [{"name": CKPT_CFG["license"]}],
        }
        with open(os.path.join(OUTPUT_DIR, "dataset-metadata.json"), "w") as f:
            json.dump(metadata, f)

        # Push: try version (dataset already exists) → fallback create (first run)
        print("\nPushing to Kaggle Dataset…")
        result = subprocess.run(
            ["kaggle", "datasets", "version",
             "-p", OUTPUT_DIR,
             "-m", f"checkpoints (latest: {ckpts[-1]})",
             "--dir-mode", "zip"],
            capture_output=True, text=True,
        )
        combined = (result.stderr + "\n" + result.stdout).lower()
        if result.returncode != 0 and any(
            kw in combined for kw in ("not found", "404", "does not exist")
        ):
            print("Dataset does not exist yet — creating (first run)…")
            result = subprocess.run(
                ["kaggle", "datasets", "create",
                 "-p", OUTPUT_DIR,
                 "--dir-mode", "zip"],
                capture_output=True, text=True,
            )

        print(result.stdout)
        if result.returncode != 0:
            print("STDERR:", result.stderr)
            raise RuntimeError(f"Kaggle CLI failed (exit {result.returncode})")

        print(f"\n✅ Done: https://www.kaggle.com/datasets/{DATASET_ID}")
        print(
            "\nSession tiếp theo: Notebook Settings → Add Data → "
            f"tìm '{CKPT_CFG['slug']}' → Add. Cell 6 sẽ tự resume."
        )